In [ ]:
# Parameters (overridden by the RunNotebook job)
src_server = ""      # serverFqdn of mirror_src_db (from GET /sqlDatabases)
src_db = ""          # databaseName of mirror_src_db
ep_server = ""       # SQL analytics endpoint host (GET /sqlEndpoints/{id}/connectionString)
ep_db = "mirror_src_db"
seed = 42
job = "workload"
rates = "10,100,1000"   # rows/s, run order is interleaved per repeat
repeats = 3
run_s = 150             # burst length per run
gap_s = 60              # idle between runs
marker_every_s = 10
poll_s = 2
tail_timeout_s = 600
idle_s = 600            # cold probes only
n_probes = 3            # cold probes only


In [ ]:
import json, struct, time, datetime, random, os, traceback, statistics
import pyodbc, notebookutils
OUT = f"/lakehouse/default/Files/mirror_lag/{job}"
os.makedirs(OUT, exist_ok=True)
def utcnow(): return datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
def conn(server, db):
    t = notebookutils.credentials.getToken("https://database.windows.net/")
    b = t.encode("utf-16-le"); ts = struct.pack(f"<I{len(b)}s", len(b), b)
    return pyodbc.connect(f"Driver={{ODBC Driver 18 for SQL Server}};Server={server};Database={db};Encrypt=yes;",
                          attrs_before={1256: ts}, autocommit=True)
def save(name, obj):
    with open(f"{OUT}/{name}", "w") as f:
        json.dump(obj, f, indent=1, default=str)
def offset_src(c, n=5):
    """seconds to add to the source clock to get notebook UTC (min-RTT sample)"""
    best = None
    for _ in range(n):
        t0 = time.time(); s = c.execute("SELECT SYSUTCDATETIME()").fetchone()[0]; t1 = time.time()
        rtt = t1 - t0; off = (t0 + t1) / 2 - s.replace(tzinfo=datetime.timezone.utc).timestamp()
        if best is None or rtt < best[0]: best = (rtt, off)
    return {"rtt_s": round(best[0], 4), "offset_s": round(best[1], 4)}
MARK_BASE = 9_000_000_000
def put_marker(src, mid, tag):
    r = src.execute("SET NOCOUNT ON; INSERT INTO dbo.table_a(id, segment, attribute_a, value, updated_at) "
                    "OUTPUT inserted.updated_at VALUES (?, 'M', ?, 0, SYSUTCDATETIME());", mid, tag).fetchone()
    return r[0]
def replication_dmvs(src):
    res = {}
    for q in ["SELECT TOP 30 * FROM sys.dm_change_feed_log_scan_sessions ORDER BY start_time DESC",
              "SELECT TOP 30 * FROM sys.dm_change_feed_errors ORDER BY entry_time DESC"]:
        try:
            cur = src.execute(q); cols = [d[0] for d in cur.description]
            res[q] = [dict(zip(cols, [str(v) for v in row])) for row in cur.fetchall()]
        except Exception as e:
            res[q] = "ERROR: " + str(e)
    return res


In [ ]:
src = conn(src_server, src_db); ep = conn(ep_server, ep_db)
mk = src.execute("SELECT ISNULL(MAX(id), ?) FROM dbo.table_a WHERE id >= ?", MARK_BASE, MARK_BASE).fetchone()[0] + 1
probes = []
for p in range(n_probes):
    t_idle = utcnow(); time.sleep(idle_s)
    off = offset_src(src)
    ct = put_marker(src, mk, f"cold_{p+1}"); ctl = ct + datetime.timedelta(seconds=off["offset_s"])
    seen = None; i = 0; c0 = time.time()
    while time.time() - c0 < tail_timeout_s:
        ps = utcnow(); i += 1
        if ep.execute(f"SELECT COUNT(*) FROM dbo.table_a WHERE id = {mk} AND {i} = {i}").fetchone()[0]: seen = ps; break
        time.sleep(poll_s)
    probes.append({"probe": p + 1, "idle_from_utc": t_idle, "idle_s": idle_s, "marker_id": mk, "commit_src_utc": ct,
                   "seen_utc": seen, "lag_s": round((seen - ctl).total_seconds(), 2) if seen else None, "polls": i, "clock": off})
    print(json.dumps(probes[-1], default=str)); mk += 1
    save("cold.json", probes)
save("dmvs_cold.json", replication_dmvs(src))
